# Lab 3 · Exercise C — A real problem: OPERA with correlated systematics
Final OPERA $\nu_\tau$-appearance analysis ([arXiv:1804.04912](https://arxiv.org), Table III): 4 channels, 10 candidates.
The uncertainties on the backgrounds and on the signal have **common sources** across channels.
We encode them in a covariance matrix $\Sigma$, factorise it as $\Sigma = A A^T$, and measure the signal strength $\mu$.

*Assumption (ours, not the collaboration's published treatment):* each source is fully correlated across the channels it touches.


In [ ]:
import ROOT
# interactive drawing in the notebook (needed on Binder)
%jsroot on
import numpy as np

MATRICOLA = 0   # TODO: write your student ID number here (digits only)
assert MATRICOLA != 0, "Set MATRICOLA first: it seeds YOUR random toy data"

ROOT.RooRandom.randomGenerator().SetSeed(MATRICOLA)
ROOT.RooMsgService.instance().setGlobalKillBelow(ROOT.RooFit.WARNING)

def check(name, ok):
    """Minimal self-check: prints PASS/FAIL and stops the notebook on FAIL."""
    print(("PASS  " if ok else "FAIL  ") + name)
    assert ok, name


In [ ]:
# OPERA final analysis, Table III (arXiv:1804.04912)
chan  = ["1h", "3h", "mu", "e"]
n_obs = np.array([6, 3, 1, 0])
sig0  = np.array([2.96, 1.83, 1.15, 0.84])        # expected signal for mu = 1
bkg0  = np.array([1.43, 0.52, 0.024, 0.035])      # expected total background

# Background uncertainty vectors by source across the 4 channels
v_charm = np.array([0.03, 0.09, 0.002, 0.007])
v_had   = np.array([0.38, 0.03, 0.0,   0.0  ])
v_las   = np.array([0.0,  0.0,  0.008, 0.0  ])
REL_SIG = 0.20                                    # 20% relative signal uncertainty in all channels


## C1 · Covariance and diagonalisation
Build $\Sigma_b = \sum_k v_k v_k^T$ over the three sources and diagonalise it with `np.linalg.eigh`.
Keep **one column per non-zero eigenvalue** ($\lambda_k > 10^{-10}$) in $A_b = U\sqrt{\Lambda}$ so that $A_b A_b^T = \Sigma_b$.


In [ ]:
Sigma_b = np.outer(v_charm, v_charm) + np.outer(v_had, v_had) + np.outer(v_las, v_las)
evals_b, U_b = np.linalg.eigh(Sigma_b)
print("Eigenvalues of Sigma_b:", evals_b)

# TODO: keep only eigenvalues > 1e-10 and compute A_b = U_b[:, keep] * np.sqrt(evals_b[keep])
A_b = None   # TODO

print("Independent nuisance parameters needed for the background:", A_b.shape[1])

# Check also the rank of a fully correlated signal (rho = 1.0)
D_sig = np.diag(REL_SIG * sig0)
Sigma_s_corr = D_sig @ np.ones((4, 4)) @ D_sig
evals_s1, U_s1 = np.linalg.eigh(Sigma_s_corr)
A_s_corr = U_s1[:, evals_s1 > 1e-10] * np.sqrt(evals_s1[evals_s1 > 1e-10])

In [ ]:
check("A_b implemented", A_b is not None)
check("A_b A_b^T reproduces Sigma_b", np.allclose(A_b @ A_b.T, Sigma_b, atol=1e-10))
check("rank = number of sources (3), not number of channels (4)", A_b.shape[1] == 3)
check("a fully correlated signal needs ONE nuisance parameter", A_s_corr.shape[1] == 1)

## C2 · The RooFit model ($\rho = 0$)
For each channel $i$, background and signal yields are linear combinations of independent unit-normal nuisance parameters $\theta_k$:
$$\beta_i = \beta_{0i} + \sum_k A^{(b)}_{ik}\theta^{(b)}_k, \qquad s_i = s_{0i} + \sum_k A^{(s)}_{ik}\theta^{(s)}_k, \qquad \lambda_i = \max(\mu s_i + \beta_i,\; 10^{-9})$$
each constrained by a 1D Gaussian $\text{Gauss}(g_k \mid \theta_k, 1)$ centered on the global observable $g_k = 0$.

Complete `beta`, `sig`, and `lam` below for $\rho = 0$ (uncorrelated signal uncertainties), then run RooStats to measure $\hat{\mu}$, the 68% interval, and $Z_0$.


In [ ]:
rho = 0.0

# Signal covariance matrix for rho = 0 and its factor A_s (given)
C_s = rho * np.ones((4, 4)) + (1.0 - rho) * np.eye(4)
Sigma_s = D_sig @ C_s @ D_sig
evals_s, U_s = np.linalg.eigh(Sigma_s)
keep_s = evals_s > 1e-10
A_s = U_s[:, keep_s] * np.sqrt(evals_s[keep_s])

# Parameter of interest (mu), nuisance parameters (theta_b, theta_s), and global observables (g_b, g_s)
mu      = ROOT.RooRealVar("mu", "signal strength", 1.0, 0.0, 5.0)
theta_b = [ROOT.RooRealVar(f"theta_b{k}", f"theta_b{k}", 0.0, -4.0, 4.0) for k in range(A_b.shape[1])]
theta_s = [ROOT.RooRealVar(f"theta_s{k}", f"theta_s{k}", 0.0, -4.0, 4.0) for k in range(A_s.shape[1])]
gb      = [ROOT.RooRealVar(f"gb{k}", f"gb{k}", 0.0, -5.0, 5.0) for k in range(A_b.shape[1])]
gs      = [ROOT.RooRealVar(f"gs{k}", f"gs{k}", 0.0, -5.0, 5.0) for k in range(A_s.shape[1])]
for g in gb + gs:
    g.setConstant(True)

# Unit width (with positive domain (0.1, 10) to avoid RooGaussian range warnings)
one    = ROOT.RooRealVar("one", "one", 1.0, 0.1, 10.0); one.setConstant(True)
constr = [ROOT.RooGaussian("c_" + th.GetName(), "c", g, th, one) for th, g in zip(theta_b + theta_s, gb + gs)]

# Build Poisson terms for the 4 channels
ns, betas, sigs, lams, poiss = [], [], [], [], []
th_b_list = ROOT.RooArgList(*theta_b)
th_s_list = ROOT.RooArgList(*theta_s)

for i, c in enumerate(chan):
    # Formula strings: e.g. "1.43 + (0.03)*@0 + (0.38)*@1 + (0.0)*@2"
    expr_b = repr(float(bkg0[i])) + "".join(f"+({float(a)!r})*@{k}" for k, a in enumerate(A_b[i]))
    expr_s = repr(float(sig0[i])) + "".join(f"+({float(a)!r})*@{k}" for k, a in enumerate(A_s[i]))

    beta = None   # TODO: ROOT.RooFormulaVar(f"beta_{c}", expr_b, th_b_list)
    sig  = None   # TODO: ROOT.RooFormulaVar(f"sig_{c}",  expr_s, th_s_list)
    lam  = None   # TODO: ROOT.RooFormulaVar(f"lam_{c}",  "max(@0*@1+@2, 1e-9)", ROOT.RooArgList(mu, sig, beta))

    n_i  = ROOT.RooRealVar(f"n_{c}", f"n_{c}", 0, 50)
    pois = ROOT.RooPoisson(f"pois_{c}", f"pois_{c}", n_i, lam)
    ns.append(n_i); betas.append(beta); sigs.append(sig); lams.append(lam); poiss.append(pois)

# Full PDF and dataset
pdf  = ROOT.RooProdPdf("opera", "opera", ROOT.RooArgList(*(poiss + constr)))
obs  = ROOT.RooArgSet(*ns)
data = ROOT.RooDataSet("data", "data", obs)
for n_i, val in zip(ns, n_obs):
    n_i.setVal(val)
data.add(obs)


In [ ]:

# Workspace and ModelConfig
w = ROOT.RooWorkspace("w_rho0")
w.Import(pdf)
w.Import(data)

mc = ROOT.RooStats.ModelConfig("mc", w)
mc.SetPdf(w.pdf("opera"))
mc.SetParametersOfInterest(ROOT.RooArgSet(w.var("mu")))
mc.SetNuisanceParameters(ROOT.RooArgSet(*(w.var(th.GetName()) for th in theta_b + theta_s)))
mc.SetGlobalObservables(ROOT.RooArgSet(*(w.var(g.GetName()) for g in gb + gs)))
mc.SetObservables(ROOT.RooArgSet(*(w.var(n_i.GetName()) for n_i in ns)))

# Fit, 68% interval, and significance Z0
w_data = w.data("data")
w_mu   = w.var("mu")
w.pdf("opera").fitTo(w_data, ROOT.RooFit.GlobalObservables(mc.GetGlobalObservables()), ROOT.RooFit.PrintLevel(-1))
mu_hat = w_mu.getVal()

plc = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
plc.SetConfidenceLevel(0.683)
iv = plc.GetInterval()
lo, hi = iv.LowerLimit(w_mu), iv.UpperLimit(w_mu)

# Fresh calculator + POI snapshot for null hypothesis (mu = 0), keeping mu >= 0 physical
w_mu.setVal(1.0)
plc_null = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
null_poi = mc.GetParametersOfInterest().snapshot()
null_poi.first().setVal(0.0)
plc_null.SetNullParameters(null_poi)
Z0 = plc_null.GetHypoTest().Significance()


In [ ]:

print(f"rho = 0.0: mu_hat = {mu_hat:.2f}  [{lo:.2f}, {hi:.2f}]   Z0 = {Z0:.2f}")
check("mu_hat close to 1.1", abs(mu_hat - 1.1) < 0.2)
check("significance of the counting model about 3.9", abs(Z0 - 3.9) < 0.4)

## C3 · Scan the correlation of the signal uncertainty
The 20% signal uncertainty comes from sources common to all channels. Compare $\rho = 0$ (independent), $0.5$, and $1$ (fully correlated across all 4 channels).


In [ ]:
results = {}
for idx, rho in enumerate((0.0, 0.5, 1.0)):
    C_s = rho * np.ones((4, 4)) + (1.0 - rho) * np.eye(4)
    Sigma_s = D_sig @ C_s @ D_sig
    evals_s, U_s = np.linalg.eigh(Sigma_s)
    keep_s = evals_s > 1e-10
    A_s = U_s[:, keep_s] * np.sqrt(evals_s[keep_s])

    mu      = ROOT.RooRealVar("mu", "signal strength", 1.0, 0.0, 5.0)
    theta_b = [ROOT.RooRealVar(f"theta_b{k}", f"theta_b{k}", 0.0, -4.0, 4.0) for k in range(A_b.shape[1])]
    theta_s = [ROOT.RooRealVar(f"theta_s{k}", f"theta_s{k}", 0.0, -4.0, 4.0) for k in range(A_s.shape[1])]
    gb      = [ROOT.RooRealVar(f"gb{k}", f"gb{k}", 0.0, -5.0, 5.0) for k in range(A_b.shape[1])]
    gs      = [ROOT.RooRealVar(f"gs{k}", f"gs{k}", 0.0, -5.0, 5.0) for k in range(A_s.shape[1])]
    for g in gb + gs:
        g.setConstant(True)

    one    = ROOT.RooRealVar("one", "one", 1.0, 0.1, 10.0); one.setConstant(True)
    constr = [ROOT.RooGaussian("c_" + th.GetName(), "c", g, th, one) for th, g in zip(theta_b + theta_s, gb + gs)]

    ns, betas, sigs, lams, poiss = [], [], [], [], []
    th_b_list = ROOT.RooArgList(*theta_b)
    th_s_list = ROOT.RooArgList(*theta_s)
    for i, c in enumerate(chan):
        expr_b = repr(float(bkg0[i])) + "".join(f"+({float(a)!r})*@{k}" for k, a in enumerate(A_b[i]))
        expr_s = repr(float(sig0[i])) + "".join(f"+({float(a)!r})*@{k}" for k, a in enumerate(A_s[i]))
        beta = ROOT.RooFormulaVar(f"beta_{c}", expr_b, th_b_list)
        sig  = ROOT.RooFormulaVar(f"sig_{c}",  expr_s, th_s_list)
        lam  = ROOT.RooFormulaVar(f"lam_{c}",  "max(@0*@1+@2, 1e-9)", ROOT.RooArgList(mu, sig, beta))
        n_i  = ROOT.RooRealVar(f"n_{c}", f"n_{c}", 0, 50)
        pois = ROOT.RooPoisson(f"pois_{c}", f"pois_{c}", n_i, lam)
        ns.append(n_i); betas.append(beta); sigs.append(sig); lams.append(lam); poiss.append(pois)

    pdf  = ROOT.RooProdPdf("opera", "opera", ROOT.RooArgList(*(poiss + constr)))
    obs  = ROOT.RooArgSet(*ns)
    data = ROOT.RooDataSet("data", "data", obs)
    for n_i, val in zip(ns, n_obs):
        n_i.setVal(val)
    data.add(obs)

    w = ROOT.RooWorkspace(f"w_scan_{idx}")
    w.Import(pdf)
    w.Import(data)

    mc = ROOT.RooStats.ModelConfig("mc", w)
    mc.SetPdf(w.pdf("opera"))
    mc.SetParametersOfInterest(ROOT.RooArgSet(w.var("mu")))
    mc.SetNuisanceParameters(ROOT.RooArgSet(*(w.var(th.GetName()) for th in theta_b + theta_s)))
    mc.SetGlobalObservables(ROOT.RooArgSet(*(w.var(g.GetName()) for g in gb + gs)))
    mc.SetObservables(ROOT.RooArgSet(*(w.var(n_i.GetName()) for n_i in ns)))

    w_data = w.data("data")
    w_mu   = w.var("mu")
    w.pdf("opera").fitTo(w_data, ROOT.RooFit.GlobalObservables(mc.GetGlobalObservables()), ROOT.RooFit.PrintLevel(-1))
    mu_hat = w_mu.getVal()

    plc = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
    plc.SetConfidenceLevel(0.683)
    iv = plc.GetInterval()
    lo, hi = iv.LowerLimit(w_mu), iv.UpperLimit(w_mu)

    w_mu.setVal(1.0)
    plc_null = ROOT.RooStats.ProfileLikelihoodCalculator(w_data, mc)
    null_poi = mc.GetParametersOfInterest().snapshot()
    null_poi.first().setVal(0.0)
    plc_null.SetNullParameters(null_poi)
    Z0 = plc_null.GetHypoTest().Significance()

    results[rho] = (mu_hat, lo, hi, Z0)
    print(f"rho = {rho:.1f} (n_theta_s = {len(theta_s)}): mu_hat = {mu_hat:.2f}  [{lo:.2f}, {hi:.2f}]  width = {hi-lo:.2f}   Z0 = {Z0:.2f}")


In [ ]:

widths = {rho: r[2] - r[1] for rho, r in results.items()}
check("the interval grows with the correlation", widths[1.0] > widths[0.0])
check("the significance of mu = 0 does not depend on the signal uncertainty", abs(results[1.0][3] - results[0.0][3]) < 0.2)


## Questions (3–4 lines each)
1. How many independent nuisance parameters ($\theta_k$) does each model need? Why is the number smaller than the number of channels?
2. By how much does the interval on $\mu$ change between $\rho=0$ and $\rho=1$? Why does $Z_0$ not change?
3. Statistical or systematic: which dominates the uncertainty on $\mu$ here? What does that say about the value of a better systematic estimate?
4. The paper quotes a larger significance ($6.1\sigma$) than the counting model. What information have we thrown away?
